<a href="https://colab.research.google.com/github/sanadv/Classical_Machine_Learning/blob/main/Classical_Machine_Leaning_Chapter_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import matplotlib.pyplot as plt

data = [
    [10, 50],
    [15, 100],
    [300, 1500],
    [320, 1600],
    [120, 500],
    [130, 550],
     [12, 55],
    [145, 600],
     [250, 1000],
    [155, 700],
     [350, 1500],
    [355, 1200]

]

# Helper function to calculate Euclidean distance
def euclidean_distance(point1, point2):
    return ((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)**0.5

# Function to plot the data and centroids
def plot_clusters(data, centroids, clusters, title):
    colors = ['r', 'g', 'b', 'c', 'm', 'y']  # Use different colors for different clusters

    # Plot data points
    for cluster_id, points in clusters.items():
        x_coords = [point[0] for point in points]
        y_coords = [point[1] for point in points]
        plt.scatter(x_coords, y_coords, c=colors[cluster_id], label=f'Cluster {cluster_id}', alpha=0.6)

    # Plot centroids
    centroid_x = [centroid[0] for centroid in centroids]
    centroid_y = [centroid[1] for centroid in centroids]
    plt.scatter(centroid_x, centroid_y, c='black', marker='x', s=100, label='Centroids')

    plt.title(title)
    plt.xlabel('Time Online (minutes)')
    plt.ylabel('Data Usage (MB)')
    plt.legend()
    plt.show()

# K-Means function
def kmeans(data, k, max_iterations=100):
    # Step 1: Initialize centroids randomly from the data points
    centroids = random.sample(data, k)

    for iteration in range(max_iterations):
        # Step 2: Assign points to the nearest centroid
        clusters = {i: [] for i in range(k)}  # Create empty clusters
        for point in data:
            distances = [euclidean_distance(point, centroid) for centroid in centroids]
            closest_centroid = distances.index(min(distances))
            clusters[closest_centroid].append(point)

        # Plot the clusters and centroids after assignment
        plot_clusters(data, centroids, clusters, title=f'Iteration {iteration + 1}: Clustering')

        # Step 3: Update centroids to the mean of assigned points
        new_centroids = []
        for i in range(k):
            if clusters[i]:  # Avoid empty cluster division by zero
                cluster_points = clusters[i]
                mean_x = sum(point[0] for point in cluster_points) / len(cluster_points)
                mean_y = sum(point[1] for point in cluster_points) / len(cluster_points)
                new_centroids.append([mean_x, mean_y])
            else:  # If a cluster is empty, reinitialize its centroid randomly
                new_centroids.append(random.choice(data))

        # Check for convergence (if centroids do not change)
        if new_centroids == centroids:
            break
        centroids = new_centroids
        print("Clusters:")
        for cluster_id, points in clusters.items():
            print(f"Cluster {cluster_id}: {points}")

        print("\nCentroids:")
        for i, centroid in enumerate(centroids):
            print(f"Centroid {i}: {centroid}")

    # Final plot of clusters
    plot_clusters(data, centroids, clusters, title='Final Clustering')

    return clusters, centroids

# Apply K-Means
k = 3  # Number of clusters
clusters, centroids = kmeans(data, k)

# Output results
print("Clusters:")
for cluster_id, points in clusters.items():
    print(f"Cluster {cluster_id}: {points}")

print("\nCentroids:")
for i, centroid in enumerate(centroids):
    print(f"Centroid {i}: {centroid}")


In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
import numpy as np

# Load the data from a CSV file
# Make sure the CSV file has columns named "Longitude" and "Latitude"
file_path = "Book2.csv"  # Update this with the actual path to your CSV file
df = pd.read_csv(file_path)

# DBSCAN parameters
epsilon = 0.025  # Adjust based on the dataset scale
min_samples = 15

# Apply DBSCAN
dbscan = DBSCAN(eps=epsilon, min_samples=min_samples)
df['Cluster'] = dbscan.fit_predict(df[['Longitude', 'Latitude']])

# Classify points
core_samples_mask = np.zeros_like(dbscan.labels_, dtype=bool)
core_samples_mask[dbscan.core_sample_indices_] = True

df['Type'] = np.where(df['Cluster'] == -1, 'Noise',
                      np.where(core_samples_mask, 'Core', 'Border'))

# Save the results to a new CSV file
output_file = "dbscan_results.csv"
df.to_csv(output_file, index=False)

print(f"Results saved to {output_file}")


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Load the dataset
data_file = 'ecoomerce.csv'
df = pd.read_csv(data_file)

# Step 2: Data preprocessing
# Dropping non-numeric columns and handling missing values
df_numeric = df.select_dtypes(include=['float64', 'int64']).dropna()

# Standardize the features
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_numeric)

# Step 3: Apply PCA
pca = PCA(n_components=0.8)
pca_result = pca.fit_transform(scaled_data)

# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_

# Step 4: Visualize explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio.cumsum(), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by Principal Components')
plt.grid(True)
plt.show()

# Show PCA results as a DataFrame
pca_df = pd.DataFrame(pca_result, columns=[f"PC{i+1}" for i in range(pca_result.shape[1])])
pca_df.head()
